# **Live-Action → Cartoon with AnimeGAN**

This notebook demonstrates how to **turn live-action photos into anime-style cartoons** using **AnimeGAN v2**, specifically **with Hayao pretrained weights**.  

We are using the [AnimeGANv2 repository](https://github.com/TachibanaYoshino/AnimeGANv2/tree/master) as a **base** for our training and inference pipeline.  

The notebook covers:

1. Loading your live-action and cartoon datasets.  
2. Preprocessing images for training.  
3. Initializing the generator with Hayao weights.  
4. Fine-tuning the model.  

> `Note:` The dataset used here is **not publicly shared** due to copyright restrictions. Images were collected manually or via web scraping.

In [1]:
# Clone the original repo so we have access to everything
!git clone https://github.com/ptran1203/pytorch-animeGAN.git
%cd pytorch-animeGAN

Cloning into 'pytorch-animeGAN'...
remote: Enumerating objects: 1895, done.
remote: Counting objects: 100% (290/290), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 1895 (delta 248), reused 244 (delta 244), pack-reused 1605 (from 2)
Receiving objects: 100% (1895/1895), 227.88 MiB | 30.00 MiB/s, done.
Resolving deltas: 100% (1038/1038), done.
/content/pytorch-animeGAN


In [13]:
# Install all libraries needed
!pip install torch torchvision tqdm opencv-python pillow

In [3]:
# Mount our Google drive for dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# We download the pre-trained weight to adjust them to our dataset
!wget https://github.com/ptran1203/pytorch-animeGAN/releases/download/v1.2/GeneratorV2_gldv2_Hayao.pt -O hayao_v2.pt

--2026-03-31 22:33:29--  https://github.com/ptran1203/pytorch-animeGAN/releases/download/v1.2/GeneratorV2_gldv2_Hayao.pt
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/364175452/51f271c7-cefe-4726-b5fc-96c54d43d25d?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-31T23%3A21%3A25Z&rscd=attachment%3B+filename%3DGeneratorV2_gldv2_Hayao.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-31T22%3A21%3A10Z&ske=2026-03-31T23%3A21%3A25Z&sks=b&skv=2018-11-09&sig=aeab1gz5%2BH7fz9qtrc6Ar%2BQLuCiJ7YNtz48PQgJpZUk%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NDk5NjcwOSwibmJmIjoxNzc0OTk2NDA5LCJwYXRoIjoicmV

In [ ]:
import zipfile
import os

# We extract our images from Drive
zip_path = "/content/drive/MyDrive/GAN/dataset_GAN_smooth.zip"
extract_path = "/content/dataset_gan"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extraído")

Since we are using someone else´s repo, there are some inconsistencies that we have to fix, like the name for the import or the module of color transfer that is missing althroughout the repo [or at least we didn´t find it], so we create and fix all those problems:

In [6]:
# make utils a module
!touch utils/__init__.py

# fix numpy
!sed -i 's/from numpy.lib.arraysetops import isin/from numpy import isin/g' models/vgg.py

# fix color transfer import
!sed -i "s/from color_transfer import color_transfer_pytorch/from utils.color_transfer import color_transfer_pytorch/g" trainer/__init__.py

# add to path
import sys
sys.path.append('/content/pytorch-animeGAN')

In [7]:
%%writefile /content/pytorch-animeGAN/utils/color_transfer.py
import torch

def color_transfer_pytorch(src, target, eps=1e-5):
    """
    Color transfer from target to src using mean/std matching (Reinhard)

    Args:
        src:    tensor [B,3,H,W] range [0,1]
        target: tensor [B,3,H,W] range [0,1]
    Returns:
        tensor [B,3,H,W] range [0,1]
    """

    assert src.shape == target.shape

    # compute mean/std per channel over spatial dims
    src_mean = src.mean(dim=[2,3], keepdim=True)
    src_std  = src.std(dim=[2,3], keepdim=True) + eps

    tgt_mean = target.mean(dim=[2,3], keepdim=True)
    tgt_std  = target.std(dim=[2,3], keepdim=True) + eps

    # normalize src and apply target stats
    out = (src - src_mean) / src_std
    out = out * tgt_std + tgt_mean

    # clamp to valid range
    return torch.clamp(out, 0.0, 1.0)

Writing /content/pytorch-animeGAN/utils/color_transfer.py


Also, we have to replicate the same structure the repo was designed for which consists of 3 folders: real, style and smooth.

We separate our dataset into those categories:

In [ ]:
import cv2
from glob import glob

# Path
cartoon_dir = "/content/dataset_gan/train/cartoon_smoothed"
style_dir = "/content/dataset_gan/train/cartoon/style"
smooth_dir = "/content/dataset_gan/train/cartoon/smooth"

# Create them if they dont exist
os.makedirs(style_dir, exist_ok=True)
os.makedirs(smooth_dir, exist_ok=True)

# Process the images
for img_path in glob(os.path.join(cartoon_dir, "*.*")):
    # Read image
    img = cv2.imread(img_path)
    h, w, c = img.shape

    # Verify 512x256
    if w < 512 or h < 256:
        print(f"Imagen muy pequeña: {img_path}")
        continue

    # Cut style | smooth
    style_img = img[:, :256, :]
    smooth_img = img[:, 256:512, :]

    # Save the images
    base_name = os.path.basename(img_path)
    cv2.imwrite(os.path.join(style_dir, base_name), style_img)
    cv2.imwrite(os.path.join(smooth_dir, base_name), smooth_img)

print("Processed images correctly")

In [ ]:
import shutil

# Paths
style_dir = "/content/dataset_gan/train/cartoon/style"
smooth_dir = "/content/dataset_gan/train/cartoon/smooth"

# Detect sub-styles
style_subdirs = [d for d in os.listdir(style_dir) if os.path.isdir(os.path.join(style_dir, d))]
smooth_files = glob(os.path.join(smooth_dir, "*.*"))
num_smooth = len(smooth_files)
print(f"Smooth has {num_smooth} images")

for sub in style_subdirs:
    sub_path = os.path.join(style_dir, sub)
    style_files = sorted(glob(os.path.join(sub_path, "*.*")))
    num_style = len(style_files)
    print(f"Sub-style {sub} has {num_style} images")

    # Remove the excess of images
    if num_style > num_smooth:
        for f in style_files[num_smooth:]:
            os.remove(f)
        print(f"Deleted {num_style - num_smooth} images from {sub}")

    # If < smooth, replicate
    elif num_style < num_smooth:
        deficit = num_smooth - num_style
        for i in range(deficit):
            src_file = style_files[i % num_style]  # Ciclar archivos existentes
            dst_file = os.path.join(sub_path, f"dup_{i}_{os.path.basename(src_file)}")
            shutil.copy2(src_file, dst_file)
        print(f"Se replicaron {deficit} imágenes en {sub}")

print("Normalized dataset")

Now we train our model, let´s learn a bit of the parameters that we can use:

- **real_image_dir**
Path to the folder containing real images (photos) used as input for the generator.
- **anime_image_dir**
Path to the folder containing anime-style images (cartoons) used as style reference.
- **test_image_dir**
Path to images used for testing/validation, to generate preview results during training without affecting the training itself.
- **model**
The model version of AnimeGAN to use, e.g.:
v1, v2 → different generator/discriminator architectures with improvements in style, quality, and stability.
- **resume_G_init**
Path to pretrained generator weights (.pt) to initialize the network before continuing training. Useful for transfer learning or continuing from a previous checkpoint.
- **batch_size**
Number of images processed in each mini-batch during training.
Larger → more stable training but higher memory usage.
Smaller → less memory but can be less stable.
- **epochs**
Number of full passes through the entire dataset.
- **init_epochs**
Number of epochs to “warm up” the generator before the discriminator starts influencing training.
  - 0 → no warm-up.
  - n>0 → train only the generator first to stabilize outputs before adversarial training.

In [12]:
!python train.py \
 --real_image_dir /content/dataset_gan/train/live_action \
 --anime_image_dir /content/dataset_gan/train/cartoon \
 --test_image_dir /content/dataset_gan/test/live_action \
 --model v2 \
 --resume_G_init /content/pytorch-animeGAN/runs_live_action_cartoon/GeneratorV2_live_action_cartoon.pt \
 --batch_size 4 \
 --epochs 10 \
 --init_epochs 0

03/31/2026 10:42:21 PM # ==== Train Config ==== #
03/31/2026 10:42:21 PM real_image_dir /content/dataset_gan/train/live_action
03/31/2026 10:42:21 PM anime_image_dir /content/dataset_gan/train/cartoon
03/31/2026 10:42:21 PM test_image_dir /content/dataset_gan/test/live_action
03/31/2026 10:42:21 PM model v2
03/31/2026 10:42:21 PM epochs 10
03/31/2026 10:42:21 PM init_epochs 0
03/31/2026 10:42:21 PM batch_size 4
03/31/2026 10:42:21 PM exp_dir runs_live_action_cartoon
03/31/2026 10:42:21 PM gan_loss lsgan
03/31/2026 10:42:21 PM resume False
03/31/2026 10:42:21 PM resume_G_init /content/pytorch-animeGAN/runs_live_action_cartoon/GeneratorV2_live_action_cartoon.pt
03/31/2026 10:42:21 PM resume_G False
03/31/2026 10:42:21 PM resume_D False
03/31/2026 10:42:21 PM device cuda
03/31/2026 10:42:21 PM use_sn False
03/31/2026 10:42:21 PM cache False
03/31/2026 10:42:21 PM amp False
03/31/2026 10:42:21 PM save_interval 1
03/31/2026 10:42:21 PM debug_samples 0
03/31/2026 10:42:21 PM num_workers 2
03